ASSIGNMENT NLP – 5 (Token Classification: POS Tagging & Chunking)


Step 1 : Install Required Libraries

In [1]:
!pip install transformers datasets seqeval torch

In [2]:
!pip install --upgrade transformers

Dataset Selection

In [3]:

! pip install datasets==2.16.1

Step 2: Import Libraries

In [4]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

Load Dataset

In [5]:
dataset = load_dataset("conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1429: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Label Categories

In [6]:
pos_labels = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

pos_labels

['"',
 "''",
 '#',
 '$',
 '(',
 ')',
 ',',
 '.',
 ':',
 '``',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'LS',
 'MD',
 'NN',
 'NNP',
 'NNPS',
 'NNS',
 'NN|SYM',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RBS',
 'RP',
 'SYM',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WP$',
 'WRB']

 Data Preprocessing(Tokenization + Alignment)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

POS Tokenization

In [34]:
def tokenize_pos(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Apply Preprocessing

In [47]:
pos_tokenized = dataset.map(tokenize_pos, batched=True)
small_train = pos_tokenized["train"].select(range(1000))
small_val = pos_tokenized["validation"].select(range(200))

In [48]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(pos_labels)
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

In [49]:
data_collator = DataCollatorForTokenClassification(tokenizer)

 Training

In [50]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer

In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator
)

In [53]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.22361260986328124, metrics={'train_runtime': 4264.4384, 'train_samples_per_second': 0.703, 'train_steps_per_second': 0.088, 'total_flos': 384785309171712.0, 'train_loss': 0.22361260986328124, 'epoch': 3.0})

In [54]:
predictions = trainer.predict(small_val)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [55]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=2)
labels = predictions.label_ids

In [56]:
true_labels = []
true_predictions = []

for pred, label in zip(preds, labels):
    true_labels.append([pos_labels[l] for l in label if l != -100])
    true_predictions.append([pos_labels[p] for p, l in zip(pred, label) if l != -100])

In [57]:
from seqeval.metrics import classification_report

In [58]:
# Inference
sentence = "John works at Google in California"
words = sentence.split()

inputs = tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True
)

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

word_ids = inputs.word_ids()

predicted_labels = []
previous_word_idx = None

for idx, word_idx in enumerate(word_ids):
    if word_idx is None:
        continue
    elif word_idx != previous_word_idx:
        predicted_labels.append(pos_labels[predictions[0][idx].item()])
    previous_word_idx = word_idx

for word, label in zip(words, predicted_labels):
    print(word, "->", label)

John -> NNP
works -> VBZ
at -> IN
Google -> NNP
in -> IN
California -> NNP


# Comparison:

| Aspect             | POS Tagging                           | Chunking                                        |
| ------------------ | ------------------------------------- | ----------------------------------------------- |
| **Definition**     | Assigns grammatical tags to each word | Groups words into meaningful phrases            |
| **Level**          | Word-level (Grammar-level)            | Phrase-level                                    |
| **Complexity**     | Easy                                  | Medium                                          |
| **Output**         | Tags like NNP, VB, JJ                 | Phrases like NP (Noun Phrase), VP (Verb Phrase) |
| **Focus**          | Identifies role of each word          | Identifies structure of sentence                |
| **Example Input**  | "John works at Google"                | Same sentence                                   |
| **Example Output** | John/NNP works/VB at/IN Google/NNP    | [John]NP [works]VP [at Google]PP                |
| **Use Case**       | Syntax analysis, grammar checking     | Information extraction, parsing                 |
| **Dependency**     | Independent task                      | Depends on POS tagging output                   |


# Summary
POS Tagging = What is each word?

Chunking = How words form meaningful groups?

# Report / Blog

**Title: Understanding POS Tagging and Chunking in NLP**

**1. Introduction**

Natural Language Processing (NLP) involves analyzing and understanding human language.

Two important steps in this process are POS Tagging and Chunking, which help machines interpret sentence structure.

**2. Differences between POS Tagging and Chunking**

**POS Tagging** assigns grammatical categories to individual words such as noun, verb, adjective, etc.

It works at the word level and is relatively simple.


**Chunking**, on the other hand, groups words into phrases like noun phrases (NP) and verb phrases (VP).

 It works at a higher level and requires understanding relationships between words.


**In short:**

POS Tagging → identifies word roles
Chunking → identifies phrase structure

**3. Challenges Faced**

a) Ambiguity in Language
Words can have multiple meanings
Example: "book" (noun or verb)

b) Context Understanding
Models may misinterpret meaning without full context

c) Data Quality Issues
Noisy or unstructured datasets reduce accuracy

d) Tagging Errors Propagation
Incorrect POS tags affect chunking results

e) Complex Sentence Structures
Long or nested sentences are harder to process

# 4. Observations and Insights
POS Tagging is foundational and easier to implement

Chunking gives deeper insights into sentence structure

Accuracy of chunking depends heavily on POS tagging

Transformer models like BERT improve both tasks significantly

Real-world text (tweets, chats) is harder than structured text

# Conclusion

POS Tagging and Chunking are essential steps in NLP pipelines.

While POS Tagging provides grammatical information at the word level, Chunking builds on it to form meaningful phrases.

Together, they help in better understanding and processing of natural language.

In [5]:
import nbformat

# Replace with your notebook filename
nb_file = "Assignment_NLP5 (4).ipynb"

# Load notebook as v4 (latest stable)
nb = nbformat.read(nb_file, as_version=4)

def fix_widgets_metadata(metadata):
    """Ensure 'widgets.state' exists or remove empty widgets."""
    if "widgets" in metadata:
        if not isinstance(metadata["widgets"], dict):
            metadata["widgets"] = {}
        if "state" not in metadata["widgets"]:
            metadata["widgets"]["state"] = {}
    return metadata

# Fix notebook-level metadata
nb.metadata = fix_widgets_metadata(nb.metadata)

# Fix each cell's metadata and outputs
for cell in nb.cells:
    cell.metadata = fix_widgets_metadata(cell.metadata)

    if "outputs" in cell:
        for output in cell.outputs:
            if "metadata" in output:
                output.metadata = fix_widgets_metadata(output.metadata)

# Save the fixed notebook
nbformat.write(nb, nb_file)
print("Notebook fully fixed! 'metadata.widgets' errors should be gone.")

Notebook fully fixed! 'metadata.widgets' errors should be gone.
